# Smoke Load: Dataset, Model, One Batch Forward (VLA-JEPA UR10e)

This notebook is the second Colab pack for the UR10e finetune. It does not
train. Its only job is to prove three things end to end on a real GPU:
the fixed data loader actually loads both dataset splits, the finetune
config actually builds a model and selectively reloads the right six
checkpoint modules, and one forward pass produces a finite loss with a
non-zero world-model loss. These three facts are what Gate 0 for this
project is waiting on.

**Required GPU: L4, not T4.** The pretrained weights alone are 5.74 GiB in
bf16. T4 has 15 GiB of VRAM total, leaving roughly 9.3 GiB for activations,
optimizer state and dataloader workers; L4 has 24 GiB and gives this smoke
test real headroom. T4 will likely still work for the batch-size-2 forward
probe below, but if it does not, that is expected -- retry on L4 before
concluding the model itself is broken.

**Do NOT use an A100.** Gate 0 is not open yet -- this notebook is what
opens it. Running on an A100 before that gate closes is out of scope for
this pack and wastes a scarce, expensive runtime on an unverified pipeline.

**Before running:** add `HF_TOKEN` to Colab Secrets (the key icon in the
left sidebar) with **write** permission on the Hugging Face token, and
enable **Notebook access** for it.

**This notebook is self-contained.** It repeats the environment build and
asset downloads from `01_colab_env.ipynb`, but every one of those steps
skips its work immediately if it is already present -- so running this
notebook in a fresh Colab session does the full setup once, and running it
in a session that already ran notebook 01 skips straight through in
seconds.

**Design principle for this notebook: every stage (S1 through S6) below
catches its own failures, records them, and lets the notebook keep going.**
A failure in one stage does not stop unrelated later stages from running --
S6 (dataloader throughput) does not need a model, so it still runs even if
S4/S5 (model build, forward pass) fail. A stage that genuinely depends on
an earlier failed stage (S3 needs S2's dataset, S5 needs S4's model) will
report `SKIPPED`, not crash the notebook. Nothing in this notebook calls
`raise` to intentionally halt -- if you see a Python traceback stop cell
execution, that is a bug in this notebook, not by design.

**If any cell times out or the network drops:** re-run that same cell.
Every download checks what is already on disk first and skips re-fetching
it, exactly as in notebook 01. Do not delete any files.

Run all cells top to bottom. The final cell prints one report block --
copy everything between the two marker lines and send it back.

## S0) Setup: repository, env-train, and the five assets

Same environment and assets as `01_colab_env.ipynb`: clone (or pull, if the
repo is already checked out from a previous notebook in this session) the
`ur10e` branch, build the `env-train` venv with `--system-site-packages`,
install `requirements.txt`, then pull the checkpoint, the two backbones,
and both dataset splits. Every step below skips its work if it is already
present at the expected size.

In [ ]:
import json
import os
import re
import subprocess
import time
import traceback
from pathlib import Path

REPO_DIR = "/content/VLA-JEPA"
CONFIG_PATH = f"{REPO_DIR}/ur10e/configs/ur10e_ft.yaml"

# Every command below that needs torch or this project's code invokes the
# literal path "/content/env-train/bin/python" directly as a subprocess --
# never a bare `import torch` in this notebook's own kernel, and never
# through an indirection variable, so every invocation stays grep-able.
REPORT = {
    "runtime_gpu": "NOT RUN",
    "runtime_vram": "NOT RUN",
    "runtime_ram": "NOT RUN",
    "repo_head_hash": "NOT RUN",
    "s0_env_status": "NOT RUN",
    "s0_assets_status": "NOT RUN",
    "s1_meta_diagnosis": "NOT RUN",
    "s2_train_status": "NOT RUN",
    "s2_heldout_status": "NOT RUN",
    "s3_sample": "NOT RUN",
    "s4_reload": "NOT RUN",
    "s5_forward": "NOT RUN",
    "s5_forward_retry_b1": "NOT RUN",
    "s6_throughput": "NOT RUN",
}
TRACEBACKS = {}

print("Report state initialized. Fields fill in as sections below run.")

In [ ]:
if os.path.isdir(os.path.join(REPO_DIR, ".git")):
    print(f"{REPO_DIR} already exists, pulling latest changes")
    pull = subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only"], capture_output=True, text=True)
    print(pull.stdout)
    print(pull.stderr)
    if pull.returncode != 0:
        print("WARNING: git pull failed, continuing with the existing checkout (see output above)")
else:
    clone = subprocess.run(
        ["git", "clone", "-b", "ur10e", "https://github.com/DuyBaoDOCer/VLA-JEPA.git", REPO_DIR],
        capture_output=True, text=True,
    )
    print(clone.stdout)
    print(clone.stderr)
    if clone.returncode != 0:
        print("WARNING: git clone failed, S0 onward will likely fail (see output above)")

head = subprocess.run(["git", "-C", REPO_DIR, "rev-parse", "HEAD"], capture_output=True, text=True)
print("HEAD:", head.stdout.strip())
REPORT["repo_head_hash"] = head.stdout.strip() if head.returncode == 0 else f"FAILED: {head.stderr.strip()}"

eol = subprocess.run(["git", "-C", REPO_DIR, "ls-files", "--eol"], capture_output=True, text=True)
crlf_lines = [line for line in eol.stdout.splitlines() if "w/crlf" in line] if eol.returncode == 0 else []
print(f"w/crlf file count: {len(crlf_lines)}")
for line in crlf_lines:
    print(" ", line)

In [ ]:
if os.path.exists("/content/env-train/bin/python"):
    print("/content/env-train already exists, skipping venv creation")
else:
    venv_create = subprocess.run(
        ["python3", "-m", "venv", "--system-site-packages", "--without-pip", "/content/env-train"],
        capture_output=True, text=True,
    )
    print(venv_create.stdout)
    print(venv_create.stderr)
    if venv_create.returncode != 0:
        print("WARNING: venv creation failed, S0 onward will likely fail (see output above)")
    else:
        print("Created venv at /content/env-train with --system-site-packages --without-pip")

In [ ]:
pt3d_install = subprocess.run(
    ["/content/env-train/bin/python", "-m", "pip", "install",
     "--ignore-requires-python", "pipablepytorch3d==0.7.6"],
    capture_output=True, text=True,
)
print(pt3d_install.stdout[-4000:])
print(pt3d_install.stderr[-4000:])

requirements_path = os.path.join(REPO_DIR, "requirements.txt")
pip_install = subprocess.run(
    ["/content/env-train/bin/python", "-m", "pip", "install", "-r", requirements_path],
    capture_output=True, text=True,
)
print(pip_install.stdout[-4000:])
print(pip_install.stderr[-4000:])

if pt3d_install.returncode == 0 and pip_install.returncode == 0:
    REPORT["s0_env_status"] = "OK: pipablepytorch3d + requirements.txt installed"
else:
    REPORT["s0_env_status"] = (
        f"FAILED: pipablepytorch3d exit={pt3d_install.returncode}, "
        f"requirements.txt exit={pip_install.returncode}"
    )
print(REPORT["s0_env_status"])

In [ ]:
from google.colab import userdata
from huggingface_hub import login, hf_hub_download, snapshot_download, HfApi

HF_USER = "DuyBao44DOCer"  # Hugging Face username -- different from the GitHub username DuyBaoDOCer


def hf_login():
    """(Re)establish the Hugging Face Hub login and return an HfApi client.

    Every download cell below calls this itself, so re-running any single
    download cell on its own still works without also re-running this cell.
    """
    login(userdata.get("HF_TOKEN"))
    client = HfApi()
    print("Logged in to Hugging Face Hub as:", client.whoami()["name"])
    return client


def snapshot_is_complete(repo_id, local_dir, repo_type="model"):
    """True if every file the Hub reports for repo_id is already on disk at the same size."""
    if not os.path.isdir(local_dir):
        return False
    try:
        if repo_type == "dataset":
            info = api.dataset_info(repo_id, files_metadata=True)
        else:
            info = api.model_info(repo_id, files_metadata=True)
    except Exception as exc:
        print(f"Could not fetch remote file list for {repo_id}, will download: {exc}")
        return False
    for sibling in info.siblings:
        if sibling.size is None:
            return False
        local_path = os.path.join(local_dir, sibling.rfilename)
        if not os.path.isfile(local_path) or os.path.getsize(local_path) != sibling.size:
            return False
    return True


def count_dataset_files(root):
    """Count parquet/side-video/wrist-video/meta files under a LeRobot dataset root.

    Only counts a "meta" match when it is NOT under a `.cache/` subtree.
    huggingface_hub's local_dir downloads leave a
    `.cache/huggingface/download/meta/*.metadata` sidecar mirror -- a
    directory that is also literally named "meta" -- which is what inflated
    the count the previous pack reported (18 instead of 6). This is a fix to
    the counting function itself, not a relaxed threshold: real dataset
    content under `meta/` is still required to be exactly 6 files.
    """
    counts = {"parquet": 0, "side_mp4": 0, "wrist_mp4": 0, "meta": 0}
    episode_indices = set()
    episode_pattern = re.compile(r"episode_(\d{6})")
    for dirpath, _, filenames in os.walk(root):
        in_cache = ".cache" in Path(dirpath).parts
        in_meta_dir = os.path.basename(dirpath) == "meta" and not in_cache
        for name in filenames:
            haystack = (dirpath + "/" + name).lower()
            match = episode_pattern.search(name)
            if match:
                episode_indices.add(int(match.group(1)))
            if in_cache:
                continue
            if name.endswith(".parquet"):
                counts["parquet"] += 1
            elif name.endswith(".mp4") and "side" in haystack:
                counts["side_mp4"] += 1
            elif name.endswith(".mp4") and "wrist" in haystack:
                counts["wrist_mp4"] += 1
            elif in_meta_dir:
                counts["meta"] += 1
    counts["episode_indices"] = episode_indices
    return counts


api = hf_login()

In [ ]:
if "api" not in globals():
    api = hf_login()

CKPT_DIR = "/content/ckpt"
CKPT_SUBPATH = "Pretrain/checkpoints/VLA-JEPA-pretrain.pt"
CKPT_EXPECTED_BYTES = 6163578232
ckpt_local_path = os.path.join(CKPT_DIR, CKPT_SUBPATH)

if os.path.isfile(ckpt_local_path) and os.path.getsize(ckpt_local_path) == CKPT_EXPECTED_BYTES:
    print(f"{ckpt_local_path} already present at expected size, skipping download")
else:
    print("Downloading checkpoint (largest single asset, ~6.16 GB)...")
    ckpt_local_path = hf_hub_download(repo_id="ginwind/VLA-JEPA", filename=CKPT_SUBPATH, local_dir=CKPT_DIR)

actual_bytes = os.path.getsize(ckpt_local_path) if os.path.isfile(ckpt_local_path) else 0
print(f"Checkpoint bytes: {actual_bytes} (expected {CKPT_EXPECTED_BYTES})")
checkpoint_status = (
    f"OK: {actual_bytes} bytes at {ckpt_local_path}" if actual_bytes == CKPT_EXPECTED_BYTES
    else f"FAILED: got {actual_bytes} bytes, expected {CKPT_EXPECTED_BYTES}"
)
print(checkpoint_status)

In [ ]:
if "api" not in globals():
    api = hf_login()

QWEN_DIR = "/content/qwen"
os.makedirs(QWEN_DIR, exist_ok=True)

if snapshot_is_complete("Qwen/Qwen3-VL-2B-Instruct", QWEN_DIR, repo_type="model"):
    print(f"{QWEN_DIR} already matches the Hub file listing at full size, skipping download")
    qwen_path = QWEN_DIR
else:
    print("Downloading Qwen/Qwen3-VL-2B-Instruct...")
    qwen_path = snapshot_download(repo_id="Qwen/Qwen3-VL-2B-Instruct", local_dir=QWEN_DIR)

qwen_file_count = sum(len(files) for _, _, files in os.walk(qwen_path))
qwen_status = f"OK: {qwen_file_count} files at {qwen_path}" if qwen_file_count > 0 else "FAILED: no files found"
print(qwen_status)

In [ ]:
if "api" not in globals():
    api = hf_login()

VJEPA2_DIR = "/content/vjepa2"
os.makedirs(VJEPA2_DIR, exist_ok=True)

if snapshot_is_complete("facebook/vjepa2-vitl-fpc64-256", VJEPA2_DIR, repo_type="model"):
    print(f"{VJEPA2_DIR} already matches the Hub file listing at full size, skipping download")
    vjepa2_path = VJEPA2_DIR
else:
    print("Downloading facebook/vjepa2-vitl-fpc64-256...")
    vjepa2_path = snapshot_download(repo_id="facebook/vjepa2-vitl-fpc64-256", local_dir=VJEPA2_DIR)

vjepa2_file_count = sum(len(files) for _, _, files in os.walk(vjepa2_path))
vjepa2_status = f"OK: {vjepa2_file_count} files at {vjepa2_path}" if vjepa2_file_count > 0 else "FAILED: no files found"
print(vjepa2_status)

In [ ]:
if "api" not in globals():
    api = hf_login()

TRAIN_DS_DIR = "/content/ds_train"
os.makedirs(TRAIN_DS_DIR, exist_ok=True)

if snapshot_is_complete(f"{HF_USER}/ur10e-cup-v21-train73", TRAIN_DS_DIR, repo_type="dataset"):
    print(f"{TRAIN_DS_DIR} already matches the Hub file listing at full size, skipping download")
else:
    print("Downloading ur10e-cup-v21-train73...")
    snapshot_download(repo_id=f"{HF_USER}/ur10e-cup-v21-train73", repo_type="dataset", local_dir=TRAIN_DS_DIR)

train_counts = count_dataset_files(TRAIN_DS_DIR)
print("Train dataset file counts:", {k: v for k, v in train_counts.items() if k != "episode_indices"})
train_ds_asset_ok = (
    train_counts["parquet"] == 73 and train_counts["side_mp4"] == 73
    and train_counts["wrist_mp4"] == 73 and train_counts["meta"] == 6
)
print("Train dataset asset check OK:", train_ds_asset_ok)

In [ ]:
if "api" not in globals():
    api = hf_login()

HELDOUT_DS_DIR = "/content/ds_heldout"
os.makedirs(HELDOUT_DS_DIR, exist_ok=True)

if snapshot_is_complete(f"{HF_USER}/ur10e-cup-v21-heldout8", HELDOUT_DS_DIR, repo_type="dataset"):
    print(f"{HELDOUT_DS_DIR} already matches the Hub file listing at full size, skipping download")
else:
    print("Downloading ur10e-cup-v21-heldout8...")
    snapshot_download(repo_id=f"{HF_USER}/ur10e-cup-v21-heldout8", repo_type="dataset", local_dir=HELDOUT_DS_DIR)

heldout_counts = count_dataset_files(HELDOUT_DS_DIR)
print("Heldout dataset file counts:", {k: v for k, v in heldout_counts.items() if k != "episode_indices"})
expected_episode_indices = set(range(73, 81))
heldout_ds_asset_ok = (
    heldout_counts["parquet"] == 8 and heldout_counts["side_mp4"] == 8
    and heldout_counts["wrist_mp4"] == 8 and heldout_counts["meta"] == 6
    and heldout_counts["episode_indices"] == expected_episode_indices
)
print("Heldout episode indices found:", sorted(heldout_counts["episode_indices"]))
print("Heldout dataset asset check OK:", heldout_ds_asset_ok)

REPORT["s0_assets_status"] = (
    f"checkpoint={checkpoint_status}; qwen={qwen_status}; vjepa2={vjepa2_status}; "
    f"train_asset_ok={train_ds_asset_ok} (parquet={train_counts['parquet']} side={train_counts['side_mp4']} "
    f"wrist={train_counts['wrist_mp4']} meta={train_counts['meta']}); "
    f"heldout_asset_ok={heldout_ds_asset_ok} (parquet={heldout_counts['parquet']} side={heldout_counts['side_mp4']} "
    f"wrist={heldout_counts['wrist_mp4']} meta={heldout_counts['meta']}, "
    f"episodes={sorted(heldout_counts['episode_indices'])})"
)
print()
print("S0 assets summary:", REPORT["s0_assets_status"])

gpu_query = subprocess.run(
    ["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
    capture_output=True, text=True,
)
REPORT["runtime_gpu"] = gpu_query.stdout.strip() if gpu_query.returncode == 0 else f"FAILED: exit {gpu_query.returncode}"
if "A100" in REPORT["runtime_gpu"].upper():
    print("=" * 60)
    print("WARNING: this runtime is an A100. Required GPU is L4 (T4 acceptable). Gate 0 is not open -- switch runtime type.")
    print("=" * 60)
print("GPU:", REPORT["runtime_gpu"])

ram = subprocess.run(["free", "-h"], capture_output=True, text=True)
REPORT["runtime_ram"] = ram.stdout.strip() if ram.returncode == 0 else f"FAILED: exit {ram.returncode}"
print(REPORT["runtime_ram"])
REPORT["runtime_vram"] = REPORT["runtime_gpu"]

## S1) Diagnose the `meta` count of 18

The previous pack's asset check counted `meta` files as 18 for both
dataset splits, even though `parquet`/side/wrist counts were exactly right
and the two splits have different episode counts (73 vs 8). Since both
splits showed the *same* 18 regardless of episode count, the number scales
with the *contents of a `meta/`-named directory*, not with episode count --
print the truth rather than guess what it is.

This cell prints the full path of every filesystem entry matching `meta`
under each dataset root (mirroring `find <path> -name "*" -path "*meta*"`),
then classifies each entry as either genuine dataset content or a
huggingface_hub `.cache/` download sidecar, and states which one it found.
This cell needs no venv -- it is plain filesystem inspection.

In [ ]:
try:
    diagnosis_lines = []
    all_real_ok = True
    for name, path in [("train", "/content/ds_train"), ("heldout", "/content/ds_heldout")]:
        print(f"--- {name} ({path}) ---")
        found = subprocess.run(
            ["find", path, "-name", "*", "-path", "*meta*"],
            capture_output=True, text=True,
        )
        print(found.stdout)
        entries = [line for line in found.stdout.splitlines() if line.strip()]
        cache_entries = [e for e in entries if "/.cache/" in e]
        real_entries = [e for e in entries if "/.cache/" not in e]
        real_files = [e for e in real_entries if not e.rstrip("/").endswith("/meta") and not e.rstrip("/") == path.rstrip("/") + "/meta"]
        print(f"{name}: total={len(entries)} under_.cache={len(cache_entries)} real={len(real_entries)}")
        all_real_ok = all_real_ok and (len(real_files) == 6)
        diagnosis_lines.append(
            f"{name}: total matches={len(entries)}, under .cache/ (sidecars)={len(cache_entries)}, "
            f"real files directly in meta/={len(real_files)}"
        )

    # This is the measured fact, not a guess: every entry is classified by
    # whether its path contains "/.cache/" -- that split is printed above
    # and in diagnosis_lines for both splits. The explanation below (Hub
    # download sidecars) is the most likely account of what a ".cache/"
    # path actually is, offered because count_dataset_files above already
    # acts on it (excludes anything under .cache/), not asserted as fact
    # independent of the counts just measured.
    conclusion = (
        "all real meta/ content is exactly 6 files per split, matching expectation"
        if all_real_ok else
        "real meta/ content is NOT exactly 6 files per split -- see the raw find output above, do not assume this is the .cache/ explanation"
    )
    REPORT["s1_meta_diagnosis"] = (
        "Measured breakdown: " + "; ".join(diagnosis_lines) + ". Conclusion: " + conclusion + ". "
        "The likely explanation for non-.cache/ noise, if any, is huggingface_hub's "
        "local_dir download sidecar mirror at '.cache/huggingface/download/meta/*.metadata' "
        "-- a directory also literally named 'meta', which is what the previous pack's "
        "counting function matched on. count_dataset_files above now excludes any path "
        "under '.cache/', which is a fix to the counting function, not a relaxed threshold."
    )
    print()
    print(REPORT["s1_meta_diagnosis"])
except Exception:
    REPORT["s1_meta_diagnosis"] = "FAILED: see Full tracebacks section"
    TRACEBACKS["s1_meta_diagnosis"] = traceback.format_exc()
    print(TRACEBACKS["s1_meta_diagnosis"])

## S2) GATE: load both dataset splits

Loads `ur10e_cup` from `/content/ds_train` and `/content/ds_heldout`
through the same `get_vla_dataset` entry point the trainer uses -- not a
lighter-weight substitute. Each split has its own try/except: a failure on
one split does not stop the other from being attempted. The held-out
split's `episode_index` values start at 73, not 0; nothing here renumbers
them if the loader chokes on that -- a failure there is reported verbatim.

The first successful load of each split writes `meta/stats_gr00t.json` into
that split's directory. That file is expected. It must never be copied
between splits -- doing so would leak held-out statistics into train,
silently and undetectably.

In [ ]:
s2_s3_script = r"""
import json
import sys
import traceback
import numpy as np
import torch.distributed as dist
from omegaconf import OmegaConf

if not dist.is_initialized():
    dist.init_process_group(backend="nccl", init_method="tcp://127.0.0.1:29502", rank=0, world_size=1)

from starVLA.dataloader.lerobot_datasets import get_vla_dataset

cfg = OmegaConf.load("CONFIG_PATH_PLACEHOLDER")
action_horizon = cfg.framework.action_model.action_horizon
video_horizon = cfg.framework.vj2_model.num_frames


def load_split(name, root_dir):
    print(f"--- loading split: {name} ({root_dir}) ---")
    data_cfg = OmegaConf.create({"data_root_dir": root_dir, "data_mix": "ur10e_cup", "with_state": True})
    try:
        ds = get_vla_dataset(data_cfg=data_cfg, action_horizon=action_horizon, video_horizon=video_horizon)
        print(f"S2_{name.upper()}_LEN={len(ds)}")
        sample = ds[0]
        print(f"S2_{name.upper()}_SAMPLE_KEYS={sorted(sample.keys())}")
        print(f"S2_{name.upper()}_OK")
        return sample
    except Exception:
        print(f"S2_{name.upper()}_FAILED")
        traceback.print_exc()
        return None


train_sample = load_split("train", "/content/ds_train")
heldout_sample = load_split("heldout", "/content/ds_heldout")

if train_sample is not None:
    print("=== S3: sample check (train split) ===")
    video = np.asarray(train_sample["video"])
    action = np.asarray(train_sample["action"])
    state = np.asarray(train_sample["state"]) if "state" in train_sample else None
    lang = train_sample["lang"]

    print(f"S3_VIDEO_SHAPE={video.shape}")
    print(f"S3_ACTION_SHAPE={action.shape}")
    print(f"S3_STATE_SHAPE={None if state is None else state.shape}")
    print(f"S3_LANG={lang!r}")

    action_first6 = action[..., :6]
    gripper_action_values = sorted(set(np.unique(action[..., 6]).tolist()))
    print(f"S3_ACTION_FIRST6_MIN={action_first6.min()}")
    print(f"S3_ACTION_FIRST6_MAX={action_first6.max()}")
    print(f"S3_GRIPPER_ACTION_VALUES={gripper_action_values}")

    if state is not None:
        state_first6 = state[..., :6]
        print(f"S3_STATE_FIRST6_MIN={state_first6.min()}")
        print(f"S3_STATE_FIRST6_MAX={state_first6.max()}")

    assert video.ndim == 5 and video.shape[0] == 2, f"expected video (V=2, T, H, W, 3), got {video.shape}"
    assert action.shape == (action_horizon, 7), f"expected action ({action_horizon}, 7), got {action.shape}"
    assert state is not None and state.shape == (1, 7), f"expected state (1, 7), got {None if state is None else state.shape}"
    print("S3_GATE_PASSED")
else:
    print("S3_SKIPPED: train split failed to load in S2")
"""
s2_s3_script = s2_s3_script.replace("CONFIG_PATH_PLACEHOLDER", CONFIG_PATH)

try:
    result = subprocess.run(
        ["/content/env-train/bin/python", "-c", s2_s3_script],
        capture_output=True, text=True, cwd=REPO_DIR,
    )
    print(result.stdout)
    print(result.stderr)

    out = result.stdout

    for split_key, split_name in [("s2_train_status", "TRAIN"), ("s2_heldout_status", "HELDOUT")]:
        if f"S2_{split_name}_OK" in out:
            len_line = next((l for l in out.splitlines() if l.startswith(f"S2_{split_name}_LEN=")), "")
            REPORT[split_key] = f"OK: {len_line}"
        elif f"S2_{split_name}_FAILED" in out:
            REPORT[split_key] = "FAILED: see Full tracebacks section"
            TRACEBACKS[split_key] = out + "\n" + result.stderr
        else:
            REPORT[split_key] = "FAILED: stage did not run to completion, see Full tracebacks section"
            TRACEBACKS[split_key] = out + "\n" + result.stderr

    if "S3_GATE_PASSED" in out:
        s3_fields = {}
        for line in out.splitlines():
            if line.startswith("S3_") and "=" in line:
                key, _, val = line.partition("=")
                s3_fields[key] = val
        REPORT["s3_sample"] = "OK: " + json.dumps(s3_fields)
    elif "S3_SKIPPED" in out:
        REPORT["s3_sample"] = "SKIPPED: S2 train split did not load"
    else:
        REPORT["s3_sample"] = "FAILED: see Full tracebacks section"
        TRACEBACKS["s3_sample"] = out + "\n" + result.stderr

    if result.returncode != 0 and "S2_TRAIN_OK" not in out and "S2_HELDOUT_OK" not in out:
        TRACEBACKS["s2_s3_full"] = out + "\n" + result.stderr
except Exception:
    REPORT["s2_train_status"] = "FAILED: notebook-side exception, see Full tracebacks section"
    REPORT["s2_heldout_status"] = "FAILED: notebook-side exception, see Full tracebacks section"
    REPORT["s3_sample"] = "FAILED: notebook-side exception, see Full tracebacks section"
    TRACEBACKS["s2_s3_full"] = traceback.format_exc()

print()
print("s2_train_status:", REPORT["s2_train_status"])
print("s2_heldout_status:", REPORT["s2_heldout_status"])
print("s3_sample:", REPORT["s3_sample"])

## S4 + S5) GATE: build the model, selectively reload, run one batch forward

Both gates share one subprocess because rebuilding the ~3.08B-parameter
model and re-reading the 6.16 GB checkpoint from local disk is cheap
(seconds, local SSD, not network) compared to a round trip through the
homeowner -- see the design principle at the top of this notebook.

**Deliberate simplification, not a spec change:** this calls
`TrainerUtils.load_pretrained_backbones`, `TrainerUtils.freeze_backbones`,
`TrainerUtils.print_trainable_parameters` and `build_param_lr_groups`
directly -- the exact functions `VLATrainer.prepare_training()` calls
internally for the reload/freeze/LR-group logic (`trainer_utils/trainer_tools.py`)
-- rather than going through the full `VLATrainer` + `Accelerate` +
DeepSpeed harness that `scripts/run_vlajepa_libero_ft.sh` uses for actual
multi-GPU training. That harness is infrastructure for distributed
training; it is not needed to prove selective reload works and one forward
pass produces a real loss on a single GPU, which is what REQ-13b and REQ-14
ask this pack to check. This is recorded in the Completion Report as a
deviation for the Contractor to review.

Those `trainer_tools.py` functions call `torch.distributed` unconditionally
(`dist.get_rank()`, `dist.barrier()`), so this script starts a trivial
single-process (`world_size=1`) NCCL process group before calling them --
without it they raise immediately since no process group exists outside of
an `accelerate launch`.

Batch size is 2, not 1 (see `ur10e_ft.yaml` and the pack notes on why B=1
cannot catch the multi-view batching regression). If this OOMs, the next
cell below is a separate retry at batch size 1 -- this cell does not
silently fall back to it.

In [ ]:
S4_S5_TEMPLATE = r"""
import io
import contextlib
import json
import math
import sys
import torch
import torch.distributed as dist
from omegaconf import OmegaConf

if not dist.is_initialized():
    dist.init_process_group(backend="nccl", init_method="tcp://127.0.0.1:29503", rank=0, world_size=1)

from starVLA.model.framework import build_framework
from starVLA.training.trainer_utils.trainer_tools import TrainerUtils, build_param_lr_groups
from starVLA.dataloader import build_dataloader

cfg = OmegaConf.load("CONFIG_PATH_PLACEHOLDER")
cfg.datasets.vla_data.per_device_batch_size = BATCH_SIZE_PLACEHOLDER
cfg.output_dir = "/content/smoke_output"
import os
os.makedirs(cfg.output_dir, exist_ok=True)

print("=== S4: build model ===")
model = build_framework(cfg)
model = model.to("cuda")

expected_loaded = sorted(p.strip() for p in cfg.trainer.reload_modules.split(",") if p.strip())
buf = io.StringIO()
with contextlib.redirect_stdout(buf):
    model = TrainerUtils.load_pretrained_backbones(
        model, cfg.trainer.pretrained_checkpoint, reload_modules=cfg.trainer.reload_modules
    )
load_log = buf.getvalue()
print(load_log)

loaded_lines = [l for l in load_log.splitlines() if l.startswith("\u2705 parameters loaded to module")]
notfound_lines = [l for l in load_log.splitlines() if "not found" in l]
loaded_paths = sorted(l.split("'")[1] for l in loaded_lines)

print(f"S4_LOADED_COUNT={len(loaded_lines)}")
print(f"S4_LOADED_PATHS={loaded_paths}")
print(f"S4_NOTFOUND_COUNT={len(notfound_lines)}")

assert len(loaded_lines) == 6, f"expected 6 loaded-module lines, got {len(loaded_lines)}: {loaded_lines}"
assert loaded_paths == expected_loaded, f"loaded paths {loaded_paths} != expected {expected_loaded}"
assert notfound_lines == [], f"unexpected not-found lines: {notfound_lines}"
for dropped in ("action_model.state_encoder", "action_model.action_encoder", "action_model.action_decoder"):
    assert not any(dropped in l for l in load_log.splitlines()), f"dropped module {dropped} appears in load log"

model = TrainerUtils.freeze_backbones(model, freeze_modules=cfg.trainer.freeze_modules)
TrainerUtils.print_trainable_parameters(model)

param_groups = build_param_lr_groups(model=model, cfg=cfg)
total_trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
group_total = 0
print("S4_LR_GROUPS:")
for g in param_groups:
    n = sum(p.numel() for p in g["params"])
    group_total += n
    print(f"  name={g['name']} lr={g['lr']} num_params={n}")
    assert n > 0, f"LR group {g['name']} is empty"

assert len(param_groups) == 3, f"expected 3 LR groups, got {len(param_groups)}: {[g['name'] for g in param_groups]}"
assert group_total == total_trainable, f"LR group total {group_total} != trainable total {total_trainable}"
print(f"S4_GROUP_TOTAL={group_total}")
print(f"S4_TRAINABLE_TOTAL={total_trainable}")
print("S4_GATE_PASSED")

print("=== S5: one batch forward ===")
dataloader = build_dataloader(cfg=cfg, dataset_py=cfg.datasets.vla_data.dataset_py)
batch = next(iter(dataloader))

torch.cuda.reset_peak_memory_stats()
try:
    with torch.autocast("cuda", dtype=torch.bfloat16):
        output_dict = model.forward(batch)
        total_loss = sum(output_dict.values())
except RuntimeError as exc:
    if "out of memory" in str(exc).lower():
        peak = torch.cuda.max_memory_allocated() / 1e9
        print(f"S5_OOM_AT_BATCH_SIZE={cfg.datasets.vla_data.per_device_batch_size}")
        print(f"S5_OOM_PEAK_VRAM_GB={peak:.2f}")
    raise

loss_val = total_loss.item()
action_loss_val = output_dict["action_loss"].item()
wm_loss_val = output_dict.get("wm_loss")
wm_loss_val = wm_loss_val.item() if wm_loss_val is not None else None
peak_vram_gb = torch.cuda.max_memory_allocated() / 1e9

print(f"S5_LOSS={loss_val}")
print(f"S5_ACTION_LOSS={action_loss_val}")
print(f"S5_WM_LOSS={wm_loss_val}")
print(f"S5_PEAK_VRAM_GB={peak_vram_gb:.2f}")
print(f"S5_BATCH_SIZE={cfg.datasets.vla_data.per_device_batch_size}")

assert math.isfinite(loss_val), f"loss is not finite: {loss_val}"
assert action_loss_val != 0, "action_loss is exactly zero"
if wm_loss_val is None or wm_loss_val == 0:
    print("S5_WM_LOSS_ZERO_BLOCKED")
    sys.exit(97)

print("S5_GATE_PASSED")
"""


def render_s4_s5_script(batch_size):
    return (
        S4_S5_TEMPLATE
        .replace("CONFIG_PATH_PLACEHOLDER", CONFIG_PATH)
        .replace("BATCH_SIZE_PLACEHOLDER", str(batch_size))
    )


try:
    result = subprocess.run(
        ["/content/env-train/bin/python", "-c", render_s4_s5_script(2)],
        capture_output=True, text=True, cwd=REPO_DIR,
    )
    print(result.stdout)
    print(result.stderr)

    out = result.stdout

    if "S4_GATE_PASSED" in out:
        lr_lines = [l for l in out.splitlines() if l.strip().startswith("name=")]
        total_lines = [l for l in out.splitlines() if l.startswith("S4_GROUP_TOTAL=") or l.startswith("S4_TRAINABLE_TOTAL=")]
        REPORT["s4_reload"] = (
            "OK: 6 modules loaded, 3 LR groups, totals match. "
            + " | ".join(lr_lines) + " || " + " | ".join(total_lines)
        )
    elif "S5_OOM_AT_BATCH_SIZE" in out:
        REPORT["s4_reload"] = "OK: model built and reloaded before the OOM happened during S5 forward"
    else:
        REPORT["s4_reload"] = "FAILED: see Full tracebacks section"
        TRACEBACKS["s4_reload"] = out + "\n" + result.stderr

    if "S5_GATE_PASSED" in out:
        fields = {}
        for line in out.splitlines():
            if line.startswith("S5_") and "=" in line:
                key, _, val = line.partition("=")
                fields[key] = val
        REPORT["s5_forward"] = "OK: " + json.dumps(fields)
    elif "S5_WM_LOSS_ZERO_BLOCKED" in out:
        REPORT["s5_forward"] = "BLOCKED: wm_loss is zero or missing -- world model is not actually engaging, see report"
        TRACEBACKS["s5_forward"] = out + "\n" + result.stderr
    elif "S5_OOM_AT_BATCH_SIZE" in out:
        peak_line = next((l for l in out.splitlines() if l.startswith("S5_OOM_PEAK_VRAM_GB=")), "")
        REPORT["s5_forward"] = f"FAILED: OOM at batch size 2 ({peak_line}); see the batch-size-1 retry cell below"
        TRACEBACKS["s5_forward"] = out + "\n" + result.stderr
    elif "S4_GATE_PASSED" not in out:
        REPORT["s5_forward"] = "SKIPPED: S4 did not pass"
    else:
        REPORT["s5_forward"] = "FAILED: see Full tracebacks section"
        TRACEBACKS["s5_forward"] = out + "\n" + result.stderr
except Exception:
    REPORT["s4_reload"] = "FAILED: notebook-side exception, see Full tracebacks section"
    REPORT["s5_forward"] = "FAILED: notebook-side exception, see Full tracebacks section"
    TRACEBACKS["s4_reload"] = traceback.format_exc()

print()
print("s4_reload:", REPORT["s4_reload"])
print("s5_forward:", REPORT["s5_forward"])

## S5 retry) Only run this cell if S5 above reported an OOM at batch size 2

This repeats S4 + S5 with `per_device_batch_size` forced to 1, purely to
tell VRAM apart from a real bug -- it does not replace the batch-size-2
result above, and it does not mean the config should be changed to batch
size 1 (see 3.2a in the pack: batch size 1 cannot exercise the multi-view
batching fix). If S5 above already passed, skip this cell.

In [ ]:
try:
    result_b1 = subprocess.run(
        ["/content/env-train/bin/python", "-c", render_s4_s5_script(1)],
        capture_output=True, text=True, cwd=REPO_DIR,
    )
    print(result_b1.stdout)
    print(result_b1.stderr)

    out_b1 = result_b1.stdout
    if "S5_GATE_PASSED" in out_b1:
        fields_b1 = {}
        for line in out_b1.splitlines():
            if line.startswith("S5_") and "=" in line:
                key, _, val = line.partition("=")
                fields_b1[key] = val
        REPORT["s5_forward_retry_b1"] = "OK: " + json.dumps(fields_b1)
    elif "S5_WM_LOSS_ZERO_BLOCKED" in out_b1:
        REPORT["s5_forward_retry_b1"] = "BLOCKED: wm_loss is zero or missing even at batch size 1"
        TRACEBACKS["s5_forward_retry_b1"] = out_b1 + "\n" + result_b1.stderr
    elif "S5_OOM_AT_BATCH_SIZE" in out_b1:
        REPORT["s5_forward_retry_b1"] = "FAILED: OOM even at batch size 1 -- this is a real capacity problem, not a batching artifact"
        TRACEBACKS["s5_forward_retry_b1"] = out_b1 + "\n" + result_b1.stderr
    else:
        REPORT["s5_forward_retry_b1"] = "FAILED: see Full tracebacks section"
        TRACEBACKS["s5_forward_retry_b1"] = out_b1 + "\n" + result_b1.stderr
except Exception:
    REPORT["s5_forward_retry_b1"] = "FAILED: notebook-side exception, see Full tracebacks section"
    TRACEBACKS["s5_forward_retry_b1"] = traceback.format_exc()

print("s5_forward_retry_b1:", REPORT["s5_forward_retry_b1"])

## S6) Dataloader throughput (no model)

Iterates the plain train dataloader for 100 batches with no model involved,
and measures frames/second (not batches/second -- the Blueprint compares
throughput by frame). Independent of S4/S5: this runs and reports even if
model build or forward failed above.

In [ ]:
s6_script = r"""
import time
import torch.distributed as dist
from omegaconf import OmegaConf
from torch.utils.data import DataLoader

if not dist.is_initialized():
    dist.init_process_group(backend="nccl", init_method="tcp://127.0.0.1:29504", rank=0, world_size=1)

from starVLA.dataloader.lerobot_datasets import get_vla_dataset, collate_fn

cfg = OmegaConf.load("CONFIG_PATH_PLACEHOLDER")
data_cfg = OmegaConf.create({
    "data_root_dir": cfg.datasets.vla_data.data_root_dir,
    "data_mix": cfg.datasets.vla_data.data_mix,
    "with_state": cfg.datasets.vla_data.with_state,
})
ds = get_vla_dataset(
    data_cfg=data_cfg,
    action_horizon=cfg.framework.action_model.action_horizon,
    video_horizon=cfg.framework.vj2_model.num_frames,
)
dl = DataLoader(ds, batch_size=cfg.datasets.vla_data.per_device_batch_size, collate_fn=collate_fn, num_workers=8)

n_batches = 100
n_frames = 0
it = iter(dl)
start = time.time()
for _ in range(n_batches):
    batch = next(it)
    for sample in batch:
        video = sample["video"]
        n_frames += video.shape[0] * video.shape[1]
elapsed = time.time() - start
fps = n_frames / elapsed if elapsed > 0 else 0.0

print(f"S6_BATCHES={n_batches}")
print(f"S6_FRAMES={n_frames}")
print(f"S6_ELAPSED_S={elapsed:.2f}")
print(f"S6_FRAMES_PER_SEC={fps:.2f}")
print("S6_GATE_PASSED")
"""
s6_script = s6_script.replace("CONFIG_PATH_PLACEHOLDER", CONFIG_PATH)

try:
    result_s6 = subprocess.run(
        ["/content/env-train/bin/python", "-c", s6_script],
        capture_output=True, text=True, cwd=REPO_DIR,
    )
    print(result_s6.stdout)
    print(result_s6.stderr)

    out_s6 = result_s6.stdout
    if "S6_GATE_PASSED" in out_s6:
        fps_line = next((l for l in out_s6.splitlines() if l.startswith("S6_FRAMES_PER_SEC=")), "")
        frames_line = next((l for l in out_s6.splitlines() if l.startswith("S6_FRAMES=")), "")
        elapsed_line = next((l for l in out_s6.splitlines() if l.startswith("S6_ELAPSED_S=")), "")
        REPORT["s6_throughput"] = f"OK: {fps_line}; {frames_line}; {elapsed_line}"
    else:
        REPORT["s6_throughput"] = "FAILED: see Full tracebacks section"
        TRACEBACKS["s6_throughput"] = out_s6 + "\n" + result_s6.stderr
except Exception:
    REPORT["s6_throughput"] = "FAILED: notebook-side exception, see Full tracebacks section"
    TRACEBACKS["s6_throughput"] = traceback.format_exc()

print("s6_throughput:", REPORT["s6_throughput"])

## S7) Final report block

Copy everything between the two marker lines below and send it back. Every
field is filled in from what actually ran above; anything skipped or
failed says so explicitly instead of being left out. Full tracebacks for
every FAILED or BLOCKED stage are included verbatim at the end.

In [ ]:
report_lines = []
report_lines.append("=========== COPY FROM HERE ===========")
report_lines.append("VLA-JEPA UR10e -- smoke load / model build / forward pass report")
report_lines.append("")
report_lines.append("-- Runtime --")
report_lines.append(f"GPU: {REPORT['runtime_gpu']}")
report_lines.append(f"RAM: {REPORT['runtime_ram']}")
report_lines.append(f"HEAD hash: {REPORT['repo_head_hash']}")
report_lines.append("")
report_lines.append("-- S0: environment + assets --")
report_lines.append(f"env-train build: {REPORT['s0_env_status']}")
report_lines.append(f"assets: {REPORT['s0_assets_status']}")
report_lines.append("")
report_lines.append("-- S1: meta count diagnosis --")
report_lines.append(REPORT["s1_meta_diagnosis"])
report_lines.append("")
report_lines.append("-- S2: dataset load (gate) --")
report_lines.append(f"train: {REPORT['s2_train_status']}")
report_lines.append(f"heldout: {REPORT['s2_heldout_status']}")
report_lines.append("")
report_lines.append("-- S3: sample check --")
report_lines.append(REPORT["s3_sample"])
report_lines.append("")
report_lines.append("-- S4: build model + selective reload (gate) --")
report_lines.append(REPORT["s4_reload"])
report_lines.append("")
report_lines.append("-- S5: one batch forward (gate) --")
report_lines.append(f"batch size 2: {REPORT['s5_forward']}")
report_lines.append(f"batch size 1 retry: {REPORT['s5_forward_retry_b1']}")
report_lines.append("")
report_lines.append("-- S6: dataloader throughput --")
report_lines.append(REPORT["s6_throughput"])
report_lines.append("")
report_lines.append("-- Full tracebacks (failed/blocked stages only) --")
if TRACEBACKS:
    for key, tb in TRACEBACKS.items():
        report_lines.append(f"### {key}")
        report_lines.append(tb)
        report_lines.append("")
else:
    report_lines.append("(none -- every stage that ran passed)")
report_lines.append("============ COPY TO HERE ============")

report_block = "\n".join(report_lines)
print(report_block)

with open("/content/smoke_load_report.txt", "w") as f:
    f.write(report_block + "\n")

print()
print("Report also written to /content/smoke_load_report.txt")